# Faruq development-v2: repair mask geometry

Jalankan hanya setelah contact sheet audit orientasi disetujui. Notebook ini memperbaiki **train/validation saja**, menurunkan box baru dari polygon mask, menjalankan audit ulang, dan mengarsipkan hasil ke Drive. Tidak ada training, inference, atau akses test.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
command = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(command)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), 'roboflow'], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)


In [ ]:
from coffee_detector.drive_project import resolve_drive_project_root

RAW_ROOT = Path('/content/faruq-segmentation-raw')
REPAIRED_ROOT = Path('/content/faruq-development-v2')
PROJECT_ROOT = resolve_drive_project_root()
AUDIT_ROOT = PROJECT_ROOT / 'evidence/faruq-mask-geometry-audit-v1'
EVIDENCE_ROOT = PROJECT_ROOT / 'evidence/faruq-mask-geometry-repair-v1'
ARCHIVE = PROJECT_ROOT / 'bundles/faruq-development-v2.tar'
RECORDS = AUDIT_ROOT / 'faruq_geometry_records.json'
assert RECORDS.is_file(), f'Record audit belum ditemukan: {RECORDS}'
EVIDENCE_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE.parent.mkdir(parents=True, exist_ok=True)

def has_coco_development(root):
    return any(
        any(path.is_file() for path in (root / split).glob('*.json'))
        for split in ('train', 'valid', 'val')
    )

if not has_coco_development(RAW_ROOT):
    from roboflow import Roboflow
    api_key = userdata.get('ROBOFLOW_API_KEY')
    rf = Roboflow(api_key=api_key)
    downloaded = rf.workspace('situju-kamkape').project('robusta_sni_dataset-hr9ci').version(1).download('coco-segmentation', location=str(RAW_ROOT))
    RAW_ROOT = Path(downloaded.location)
assert has_coco_development(RAW_ROOT), RAW_ROOT
print('RAW     :', RAW_ROOT)
print('REPAIRED:', REPAIRED_ROOT)


In [ ]:
import json
from coffee_detector.repair_faruq_mask_geometry import repair_faruq_mask_geometry

repair = repair_faruq_mask_geometry(RAW_ROOT, RECORDS, REPAIRED_ROOT)
assert repair['training_executed'] is False
assert repair['inference_executed'] is False
assert repair['test_images_accessed'] is False
assert repair['training_ready'] is False
print(json.dumps(repair, indent=2, ensure_ascii=False))


In [ ]:
from coffee_detector.audit_faruq_mask_geometry import audit_faruq_mask_geometry

POST_AUDIT_ROOT = EVIDENCE_ROOT / 'post_repair_audit'
post = audit_faruq_mask_geometry(REPAIRED_ROOT, POST_AUDIT_ROOT, score_long_side=192, min_improvement=0.02, contact_sheet_limit=24)
assert post['test_images_accessed'] is False
assert post['training_executed'] is False
print(json.dumps(post, indent=2, ensure_ascii=False))
for name in ('faruq_geometry_repair_summary.json', 'faruq_geometry_repair_manifest.json', 'faruq_geometry_repair_quarantine.json'):
    shutil.copy2(REPAIRED_ROOT / name, EVIDENCE_ROOT / name)


In [ ]:
import tarfile

if not ARCHIVE.is_file():
    temporary = Path('/content/faruq-development-v2.tar')
    with tarfile.open(temporary, 'w') as archive:
        archive.add(REPAIRED_ROOT, arcname='faruq-development-v2')
    shutil.copy2(temporary, ARCHIVE)
    temporary.unlink()
print('ARCHIVE:', ARCHIVE, f'({ARCHIVE.stat().st_size / 1024**2:.1f} MB)')


In [ ]:
from IPython.display import display
from PIL import Image

sheet = POST_AUDIT_ROOT / 'flagged_orientation_contact_sheet.jpg'
if sheet.is_file():
    display(Image.open(sheet))
else:
    print('PASS GEOMETRI: audit ulang tidak menghasilkan kasus flagged.')
print('REPAIR SUMMARY:', EVIDENCE_ROOT / 'faruq_geometry_repair_summary.json')
print('POST AUDIT    :', POST_AUDIT_ROOT / 'faruq_geometry_audit_summary.json')
print('Kirim dua summary tersebut. Jangan training sebelum leakage dan contact sheet dinilai.')
